# Build a mini-VLA from Scratch

**Lecture 2, Part 2 — Vizuara Modern Robot Learning Bootcamp**

We build a complete **Vision-Language-Action** model in ~150 lines of PyTorch and train it on the **PushT** environment:

1. **TinyCNN** — vision encoder (image → 128-d)
2. **TextEncoderGRU** — language encoder (sentence → 128-d)
3. **StateEncoderMLP** — proprioception encoder (robot state → 128-d)
4. **FusionMLP** — duct-tape fusion (concatenate + MLP → 128-d)
5. **DiffusionHead** — action generation (128-d → trajectory)

We train it with **4 different language instructions** ("push the block left/right/forward/back"), watch `nn.Embedding` learn what words mean from scratch, then test where it **fails**.

**Key design choice:** We deliberately give the model only the **agent position** (2-d) as state — NOT the block position. This forces the model to **look at the image** to find the block and **read the language instruction** to know which direction to push. If we gave it the block position, it could cheat by computing the direction from numbers alone!

Credit: Architecture inspired by [keivalya/mini-vla](https://github.com/keivalya/mini-vla)

---

In [ ]:
!pip install -q "pymunk<7" gym-pusht torch torchvision numpy matplotlib scikit-learn 2>&1 | tail -3

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from PIL import Image
from IPython.display import display, HTML
import os, copy

# PushT uses pygame — headless rendering for Colab
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["PYGAME_HIDE_SUPPORT_PROMPT"] = "1"

import gym_pusht  # registers gym_pusht/PushT-v0 with gymnasium

# Vizuara color palette
ACCENT = '#D97757'
BLUE   = '#5B8CB8'
TEAL   = '#7DA488'
PURPLE = '#9B7EC8'
WARM   = '#C4956A'
RED    = '#D4543A'
BG     = '#FAF8F5'

plt.rcParams['figure.facecolor'] = BG
plt.rcParams['axes.facecolor'] = BG
plt.rcParams['font.size'] = 12

torch.manual_seed(42)
np.random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

---
## Step 1: Meet the PushT Environment

[PushT](https://github.com/huggingface/gym-pusht) is a 2D environment from the Diffusion Policy paper. A circular **agent** (end-effector) must push a **T-shaped block** (blue) to match a **target T** (green).

**Why PushT?** The goal is immediately visible — you can see the green target and the blue block at a glance. No hidden goals!

**Observation (state mode):** agent position (2) + block pose (3: x, y, angle) = 5-d  
**Action:** 2-d — target (x, y) position for the agent

In [ ]:
env = gym.make("gym_pusht/PushT-v0", obs_type="state", render_mode="rgb_array")
obs, info = env.reset(seed=42)
frame = env.render()

print(f"Frame shape: {frame.shape}")
print(f"Observation shape: {obs.shape}, values: {obs.round(1)}")
print(f"  obs[:2]  = agent pos:  {obs[:2].round(1)}")
print(f"  obs[2:4] = block pos:  {obs[2:4].round(1)}")
print(f"  obs[4]   = block angle: {obs[4]:.2f} rad")
print(f"\nAction space: {env.action_space}")
print(f"  Action = target (x, y) position for the agent in [0, 512]")

# Goal is in the info dict, fixed at [256, 256, pi/4]
goal_pose = info["goal_pose"]
print(f"\nGoal pose (from info): {goal_pose.round(1)}")
print(f"  Goal is FIXED at center of arena — block starts in random positions")
print(f"  Direction from block to goal varies per episode => 4 language instructions")

# Show the frame
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(frame)
ax.set_title("PushT: Push Blue T onto Green Target", fontsize=14, color=ACCENT, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()
env.close()

---
## Step 2: The Expert Policy + Language from Goal Direction

We write a scripted expert that pushes the T-block toward the target:
1. **Approach:** Move agent behind the block (opposite side from target)
2. **Push:** Drive agent through block toward target

We derive the language instruction from the **direction** between block center and target center:
- Target is right of block → *"push the block right"*
- Target is left → *"push the block left"*  
- Target is up (toward top of screen) → *"push the block forward"*
- Target is down → *"push the block back"*

This gives us 4 distinct instructions. The model **must** learn what "left" vs "right" means to predict the correct action.

In [ ]:
def get_direction_label(block_pos, goal_pos):
    """Classify goal direction relative to block into 4 instructions.

    PushT coordinate system:
      +x = right in rendered image, +y = down in rendered image
    """
    diff = goal_pos[:2] - block_pos[:2]  # [dx, dy]
    if abs(diff[0]) > abs(diff[1]):
        return "push the block right" if diff[0] > 0 else "push the block left"
    else:
        return "push the block back" if diff[1] > 0 else "push the block forward"


def expert_push_policy(obs, goal_pos):
    """Scripted 2-phase push expert for PushT.

    Phase 1: Position agent behind block (opposite side from target)
    Phase 2: Push through block toward target
    Action: target (x, y) position for the agent in [0, 512]
    """
    agent_pos = obs[:2]
    block_pos = obs[2:4]

    # Direction from block to goal
    push_dir = goal_pos[:2] - block_pos
    push_dist = np.linalg.norm(push_dir)
    push_dir_norm = push_dir / max(push_dist, 1e-6)

    # Approach point: close behind the block
    approach = block_pos - push_dir_norm * 20

    # Check if agent is roughly in position behind block
    agent_to_approach = np.linalg.norm(agent_pos - approach)

    if agent_to_approach > 15:
        # Phase 1: move to approach point
        target = approach
    else:
        # Phase 2: push — aim well past the block toward goal
        overshoot = goal_pos[:2] + push_dir_norm * 100
        target = overshoot

    return np.clip(target, 0, 512).astype(np.float32)


# Demo: run episodes until we see all 4 directions, record video frames
env = gym.make("gym_pusht/PushT-v0", obs_type="state", render_mode="rgb_array")
demo_data = {}  # label -> (frames_list, init_frame, final_frame)
DEMO_STEPS = 150

for seed in range(300):
    obs, info = env.reset(seed=seed)
    block_pos = obs[2:4]
    goal_pos = info["goal_pose"][:2]
    label = get_direction_label(block_pos, goal_pos)
    if label in demo_data:
        continue

    frames = [env.render()]
    for _ in range(DEMO_STEPS):
        action = expert_push_policy(obs, info["goal_pose"])
        obs, _, _, _, info = env.step(action)
        frames.append(env.render())

    demo_data[label] = frames
    if len(demo_data) == 4:
        break

env.close()

# Plot: initial and final frames for each direction
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
colors_map = {"push the block right": ACCENT, "push the block left": BLUE,
              "push the block forward": TEAL, "push the block back": PURPLE}

for col, (label, frames) in enumerate(demo_data.items()):
    c = colors_map[label]
    axes[0, col].imshow(frames[0])
    axes[0, col].set_title(f'"{label}"', fontsize=12, color=c, fontweight='bold')
    axes[0, col].set_ylabel("Before", fontsize=11)
    axes[0, col].set_xticks([]); axes[0, col].set_yticks([])

    axes[1, col].imshow(frames[-1])
    axes[1, col].set_ylabel(f"After {DEMO_STEPS} steps", fontsize=11)
    axes[1, col].set_xticks([]); axes[1, col].set_yticks([])

plt.suptitle("Expert Policy: 4 Push Directions (Before / After)",
             fontsize=16, color=ACCENT, fontweight='bold')
plt.tight_layout()
plt.show()
print(f"Collected demo episodes for: {list(demo_data.keys())}")

# ── Video recordings ──
import imageio, base64, io
from IPython.display import HTML

html_parts = ['<div style="display:flex; flex-wrap:wrap; gap:10px; justify-content:center;">']
for label, frames in demo_data.items():
    c = colors_map[label]
    # Subsample frames for smaller GIF (every 5th frame)
    sub_frames = frames[::5]
    buf = io.BytesIO()
    imageio.mimsave(buf, sub_frames, format='GIF', duration=100, loop=0)
    b64 = base64.b64encode(buf.getvalue()).decode()
    html_parts.append(f'''
    <div style="text-align:center;">
      <div style="color:{c}; font-weight:bold; font-size:13px; margin-bottom:4px;">{label}</div>
      <img src="data:image/gif;base64,{b64}" style="width:240px; border:2px solid {c}; border-radius:8px;"/>
    </div>''')
html_parts.append('</div>')
display(HTML(''.join(html_parts)))

---
## Step 3: Adding Language — Vocabulary & Tokenization

Our 4 instructions use only **12 unique words**. We build a tiny vocabulary and a simple tokenizer.

This is the same idea as real VLAs — just with 12 words instead of 256,000.

In [ ]:
# Our 4 instructions
instructions = [
    "push the block right",
    "push the block left",
    "push the block forward",
    "push the block back",
]

# Build vocabulary from all words
all_words = set()
for instr in instructions:
    all_words.update(instr.split())
all_words = sorted(all_words)

vocab = {word: i for i, word in enumerate(all_words)}
vocab["<pad>"] = len(vocab)
VOCAB_SIZE = len(vocab)

print("Vocabulary:")
for word, idx in vocab.items():
    print(f"  '{word}' → {idx}")

MAX_LEN = 5  # longest instruction is 4 words, pad to 5

def tokenize(sentence, vocab, max_len=MAX_LEN):
    """Convert sentence to fixed-length token ID list."""
    tokens = [vocab.get(w, vocab["<pad>"]) for w in sentence.lower().split()]
    tokens = tokens[:max_len] + [vocab["<pad>"]] * max(0, max_len - len(tokens))
    return tokens

# Show tokenization
print(f"\nTokenized examples (max_len={MAX_LEN}):")
for instr in instructions:
    toks = tokenize(instr, vocab)
    print(f"  '{instr}' → {toks}")

---
## Step 4: How `nn.Embedding` Works (Deep Dive)

`nn.Embedding(vocab_size, d_word)` is literally a **lookup table** — a matrix of shape `[vocab_size, d_word]`. When you pass in token ID `3`, it returns row 3 of the matrix.

**At initialization, these vectors are random noise.** The word "right" and "left" have no meaningful difference — they're just random 64-d vectors. Training will teach the network what each word should mean, purely from the action prediction loss.

In [ ]:
# Create a standalone embedding layer to inspect
demo_embed = nn.Embedding(VOCAB_SIZE, 64)

print(f"Embedding weight matrix shape: {demo_embed.weight.shape}")
print(f"  → {VOCAB_SIZE} words × 64 dimensions\n")

# Show the raw embedding vectors at initialization
id2word = {v: k for k, v in vocab.items()}
print("Embedding vectors at initialization (first 6 dims):")
print("-" * 55)
for idx in range(VOCAB_SIZE):
    word = id2word[idx]
    vec = demo_embed.weight[idx].detach().numpy()
    print(f"  {word:>8s} (id={idx}): [{vec[0]:+.3f}, {vec[1]:+.3f}, {vec[2]:+.3f}, {vec[3]:+.3f}, {vec[4]:+.3f}, {vec[5]:+.3f}, ...]")

print(f"\nThese numbers are RANDOM. 'right' and 'left' look identical — just noise.")

# Show forward pass
token_ids = torch.tensor([tokenize("push the block right", vocab)])
embedded = demo_embed(token_ids)
print(f"\nForward pass: token_ids {token_ids[0].tolist()} → embedded shape {list(embedded.shape)}")
print(f"  Each token ID → its row from the weight matrix. That's ALL embedding does.")

# Show cosine similarity between direction words
with torch.no_grad():
    right_vec = demo_embed.weight[vocab["right"]]
    left_vec = demo_embed.weight[vocab["left"]]
    fwd_vec = demo_embed.weight[vocab["forward"]]
    cos_rl = F.cosine_similarity(right_vec.unsqueeze(0), left_vec.unsqueeze(0)).item()
    cos_rf = F.cosine_similarity(right_vec.unsqueeze(0), fwd_vec.unsqueeze(0)).item()
    print(f"\nCosine similarity (before training):")
    print(f"  right vs left:    {cos_rl:+.3f}  (should be far apart after training)")
    print(f"  right vs forward: {cos_rf:+.3f}  (should also differ after training)")
    print(f"  → All similarities are random. No structure yet.")

---
## Step 5: How GRU Processes Language (Deep Dive)

The GRU reads word embeddings **one at a time**, updating a hidden state at each step. After the last word, the hidden state is a **single 128-d vector** summarizing the entire sentence.

Think of it as a bottleneck: "push the block right" (4 words × 64-d) → 1 × 128-d vector.

In [ ]:
# Step through GRU word by word
demo_gru = nn.GRU(input_size=64, hidden_size=128, batch_first=True)
sentence = "push the block right"
token_ids = torch.tensor([tokenize(sentence, vocab)])
embedded = demo_embed(token_ids)  # [1, 5, 64]

# Process word by word and capture hidden states
hidden = torch.zeros(1, 1, 128)
hidden_states = [hidden[0, 0].detach().numpy().copy()]
words = sentence.split() + ["<pad>"]

print(f"Processing: '{sentence}'\n")
for t in range(embedded.shape[1]):
    word_emb = embedded[:, t:t+1, :]  # [1, 1, 64]
    _, hidden = demo_gru(word_emb, hidden)
    hidden_states.append(hidden[0, 0].detach().numpy().copy())
    h_norm = np.linalg.norm(hidden[0, 0].detach().numpy())
    print(f"  Step {t}: '{words[t]:>8s}' → hidden norm = {h_norm:.3f}")

print(f"\nFinal hidden state: 128-d vector (norm = {np.linalg.norm(hidden_states[-1]):.3f})")
print(f"This single vector must encode EVERYTHING about the instruction.")

# Visualize hidden state evolution
fig, ax = plt.subplots(figsize=(12, 3))
h_matrix = np.array(hidden_states)  # [6, 128]
im = ax.imshow(h_matrix.T[:32, :], aspect='auto', cmap='RdBu_r', vmin=-0.5, vmax=0.5)
ax.set_xticks(range(len(hidden_states)))
ax.set_xticklabels(["init"] + words, fontsize=11)
ax.set_ylabel("Hidden dims (first 32)")
ax.set_title("GRU Hidden State After Each Word", fontsize=14, color=ACCENT, fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()
print("Each column shows the hidden state after processing that word.")
print("Notice how the state changes most at the LAST content word ('right') — that's the direction info.")

---
## Step 6: Collect Training Data

We run the expert policy for **200 episodes** (50 per direction) and store:
- **Image:** 64×64 rendered frame from the environment
- **Tokens:** tokenized instruction (determined by goal direction)
- **State:** 2-d agent position only (NOT block position — we want the model to rely on the image and language!)
- **Actions:** 8-step trajectory from the expert

**Why only agent position?** The goal is fixed at (256, 256). If we gave the model the block position, it could trivially compute "block is left of goal → push right" without ever learning what the words mean. By hiding the block position, the model **must** use the image (to see the block) and the text (to know the direction).

In [ ]:
STATE_DIM = 2    # agent_x, agent_y ONLY (no block position — force model to use vision + language)
ACTION_DIM = 2   # target x, y for agent
HORIZON = 8      # predict 8 future actions

def collect_data(n_episodes=200, seed_start=1000):
    """Collect expert demonstrations from PushT."""
    env = gym.make("gym_pusht/PushT-v0", obs_type="state", render_mode="rgb_array")

    images_list, tokens_list, states_list, actions_list, labels_list = [], [], [], [], []
    direction_counts = {"push the block right": 0, "push the block left": 0,
                        "push the block forward": 0, "push the block back": 0}
    target_per_dir = n_episodes // 4

    seed = seed_start
    while min(direction_counts.values()) < target_per_dir:
        obs, info = env.reset(seed=seed)
        seed += 1

        block_pos = obs[2:4]
        goal_pos = info["goal_pose"][:2]
        label = get_direction_label(block_pos, goal_pos)
        if direction_counts[label] >= target_per_dir:
            continue

        # Render initial frame, resize to 64x64
        frame = env.render()
        img_pil = Image.fromarray(frame).resize((64, 64), Image.BILINEAR)
        img_tensor = torch.tensor(np.array(img_pil), dtype=torch.float32).permute(2, 0, 1) / 255.0

        # Store ONLY agent position (2-d) — NOT block position!
        state = torch.tensor(obs[:STATE_DIM], dtype=torch.float32)

        # Collect HORIZON action steps from expert
        traj = []
        for _ in range(HORIZON):
            action = expert_push_policy(obs, info["goal_pose"])
            traj.append(torch.tensor(action[:ACTION_DIM], dtype=torch.float32))
            obs, _, _, _, info = env.step(action)

        images_list.append(img_tensor)
        tokens_list.append(torch.tensor(tokenize(label, vocab)))
        states_list.append(state)
        actions_list.append(torch.stack(traj))
        labels_list.append(label)
        direction_counts[label] += 1

    env.close()

    return {
        'images': torch.stack(images_list),    # [N, 3, 64, 64]
        'texts': torch.stack(tokens_list),      # [N, MAX_LEN]
        'states': torch.stack(states_list),     # [N, 2]  ← agent position only!
        'actions': torch.stack(actions_list),   # [N, 8, 2]
        'labels': labels_list,
    }

print("Collecting 200 expert demonstrations (50 per direction)...")
data = collect_data(200)

print(f"\nDataset shapes:")
for k, v in data.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k:>8s}: {list(v.shape)}")

print(f"\nState = agent position only: {list(data['states'].shape)}")
print(f"  The model does NOT see block position — it must use the IMAGE to find the block")
print(f"  and the LANGUAGE to know which direction to push!")

print(f"\nDirection distribution:")
for label in sorted(set(data['labels'])):
    count = data['labels'].count(label)
    print(f"  '{label}': {count} episodes")

# Show sample images from each direction
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
seen = set()
for i, label in enumerate(data['labels']):
    if label not in seen and len(seen) < 4:
        seen.add(label)
        ax = axes[len(seen)-1]
        img = data['images'][i].permute(1, 2, 0).numpy()
        ax.imshow(img)
        ax.set_title(f'"{label}"', fontsize=11, color=ACCENT, fontweight='bold')
        ax.axis('off')
plt.suptitle("Sample Training Images (64x64)", fontsize=14, color=ACCENT, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Visualize training episodes: image + waypoints + GIF rollout ──
# We re-collect a few full episodes for visualization (the stored state is only 2-d)

import imageio, base64, io

ROLLOUT_STEPS_PER_WAYPOINT = 15  # steps to execute each waypoint action

def rollout_waypoints(waypoints, seed, steps_per_wp=ROLLOUT_STEPS_PER_WAYPOINT):
    """Execute waypoints in the env and record frames."""
    env = gym.make("gym_pusht/PushT-v0", obs_type="state", render_mode="rgb_array")
    obs, info = env.reset(seed=seed)
    frames = [env.render()]
    for wp in waypoints:
        for _ in range(steps_per_wp):
            obs, _, _, _, info = env.step(np.array(wp, dtype=np.float32))
            frames.append(env.render())
    env.close()
    return frames

def frames_to_gif_b64(frames, every_n=3, duration=100):
    """Convert frames to base64-encoded GIF."""
    sub = frames[::every_n]
    buf = io.BytesIO()
    imageio.mimsave(buf, sub, format='GIF', duration=duration, loop=0)
    return base64.b64encode(buf.getvalue()).decode()

# Collect 4 training episodes (one per direction) with full obs + video
env_viz = gym.make("gym_pusht/PushT-v0", obs_type="state", render_mode="rgb_array")
viz_samples = []  # (label, full_obs, goal_pos, waypoints, img, seed)
seen_viz = set()
seed = 1000

while len(viz_samples) < 4:
    obs, info = env_viz.reset(seed=seed)
    cur_seed = seed
    seed += 1
    label = get_direction_label(obs[2:4], info["goal_pose"][:2])
    if label in seen_viz:
        continue
    seen_viz.add(label)
    frame = env_viz.render()
    img_pil = Image.fromarray(frame).resize((64, 64), Image.BILINEAR)
    full_obs = obs.copy()
    goal_pos = info["goal_pose"][:2].copy()
    waypoints = []
    for _ in range(HORIZON):
        action = expert_push_policy(obs, info["goal_pose"])
        waypoints.append(action[:ACTION_DIM].copy())
        obs, _, _, _, info = env_viz.step(action)
    viz_samples.append((label, full_obs, goal_pos, np.array(waypoints), np.array(img_pil), cur_seed))
env_viz.close()

# ── Row 1: Static plots (image + waypoints) ──
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for col, (label, full_obs, goal_pos, waypoints, img, sd) in enumerate(viz_samples):
    c = colors_map[label]
    agent_pos = full_obs[:2]
    block_pos = full_obs[2:4]

    axes[0, col].imshow(img)
    axes[0, col].set_title(f'"{label}"', fontsize=12, color=c, fontweight='bold')
    axes[0, col].text(0.02, 0.98, f"agent: ({agent_pos[0]:.0f}, {agent_pos[1]:.0f})  ← model sees this",
                       transform=axes[0, col].transAxes, fontsize=9, va='top', color=BLUE, fontweight='bold')
    axes[0, col].text(0.02, 0.86, f"block: ({block_pos[0]:.0f}, {block_pos[1]:.0f})  ← hidden from model",
                       transform=axes[0, col].transAxes, fontsize=9, va='top', color='gray', fontweight='bold')
    axes[0, col].axis('off')

    ax = axes[1, col]
    ax.plot(agent_pos[0], agent_pos[1], 'o', color=BLUE, markersize=12, label='Agent', zorder=5)
    ax.plot(block_pos[0], block_pos[1], 's', color='gray', markersize=12, label='Block', zorder=5)
    ax.plot(256, 256, '*', color=TEAL, markersize=15, label='Goal', zorder=5)
    for t in range(len(waypoints)):
        alpha = 0.4 + 0.6 * (t / (len(waypoints) - 1))
        ax.plot(waypoints[t, 0], waypoints[t, 1], 'D', color=c, markersize=8, alpha=alpha, zorder=4)
        if t > 0:
            ax.plot([waypoints[t-1, 0], waypoints[t, 0]], [waypoints[t-1, 1], waypoints[t, 1]],
                    '-', color=c, alpha=alpha, linewidth=2)
    ax.plot([agent_pos[0], waypoints[0, 0]], [agent_pos[1], waypoints[0, 1]],
            '--', color=c, alpha=0.4, linewidth=1)
    ax.set_xlim(0, 512); ax.set_ylim(512, 0)
    ax.set_aspect('equal')
    ax.set_title("8 Expert Waypoints", fontsize=11, color=c, fontweight='bold')
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(True, alpha=0.2)

plt.suptitle("Training Data: What the Model Learns to Predict",
             fontsize=16, color=ACCENT, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Row 2: GIF rollouts of expert waypoints ──
print("\nExpert rollout recordings (executing the 8 stored waypoints in the environment):")
html_parts = ['<div style="display:flex; flex-wrap:wrap; gap:12px; justify-content:center; margin-top:8px;">']
for label, full_obs, goal_pos, waypoints, img, sd in viz_samples:
    c = colors_map[label]
    frames = rollout_waypoints(waypoints, seed=sd)
    b64 = frames_to_gif_b64(frames)
    html_parts.append(f'''
    <div style="text-align:center;">
      <div style="color:{c}; font-weight:bold; font-size:13px; margin-bottom:4px;">{label}</div>
      <img src="data:image/gif;base64,{b64}" style="width:220px; border:2px solid {c}; border-radius:8px;"/>
      <div style="font-size:11px; color:#8c7e6f; margin-top:2px;">Expert: {len(waypoints)} waypoints</div>
    </div>''')
html_parts.append('</div>')
display(HTML(''.join(html_parts)))

---
## Step 7: Build All Encoders + Diffusion Head

Same architecture as before: TinyCNN (vision), GRU (language), MLP (state), and FusionMLP. The diffusion head learns to denoise action trajectories conditioned on the fused 128-d vector.

In [ ]:
# ── Vision Encoder: TinyCNN ──────────────────────────────────────
class ImageEncoderTinyCNN(nn.Module):
    def __init__(self, d_model=128):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=5, stride=2, padding=2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1)
        self.proj = nn.Linear(128, d_model)
        self.ln = nn.LayerNorm(d_model)

    def forward(self, x):              # x: [B, 3, H, W]
        x = F.relu(self.conv1(x))      # -> [B, 32, H/2, W/2]
        x = F.relu(self.conv2(x))      # -> [B, 64, H/4, W/4]
        x = F.relu(self.conv3(x))      # -> [B, 128, H/8, W/8]
        x = x.mean(dim=[2, 3])         # Global Average Pool -> [B, 128]
        return self.ln(self.proj(x))   # -> [B, d_model]

# ── Language Encoder: GRU ────────────────────────────────────────
class TextEncoderTinyGRU(nn.Module):
    def __init__(self, vocab_size, d_word=64, d_model=128):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_word)
        self.gru = nn.GRU(d_word, d_model, batch_first=True)
        self.ln = nn.LayerNorm(d_model)

    def forward(self, token_ids):       # [B, seq_len]
        x = self.embed(token_ids)       # -> [B, seq_len, d_word]
        _, h_last = self.gru(x)         # h_last: [1, B, d_model]
        return self.ln(h_last[0])       # -> [B, d_model]

# ── State Encoder: MLP ───────────────────────────────────────────
class StateEncoderMLP(nn.Module):
    def __init__(self, state_dim, d_model=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, d_model),
        )
        self.ln = nn.LayerNorm(d_model)

    def forward(self, s):               # [B, state_dim]
        return self.ln(self.net(s))     # -> [B, d_model]

# ── Fusion: Concatenation + MLP ──────────────────────────────────
class FusionMLP(nn.Module):
    def __init__(self, d_model=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3 * d_model, d_model),   # 384 -> 128
            nn.ReLU(),
            nn.Linear(d_model, d_model),       # 128 -> 128
        )
        self.ln = nn.LayerNorm(d_model)

    def forward(self, img_tok, txt_tok, state_tok):
        x = torch.cat([img_tok, txt_tok, state_tok], dim=-1)  # [B, 384]
        return self.ln(self.net(x))                            # [B, 128]

# ── Simplified Diffusion Head ────────────────────────────────────
class SimpleDiffusionHead(nn.Module):
    def __init__(self, action_dim, horizon, cond_dim=128):
        super().__init__()
        self.action_dim = action_dim
        self.horizon = horizon
        flat_dim = action_dim * horizon
        self.noise_pred = nn.Sequential(
            nn.Linear(flat_dim + cond_dim + 1, 256),  # +1 for timestep
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, flat_dim),
        )
        self.T = 20  # diffusion timesteps
        betas = torch.linspace(1e-4, 0.02, self.T)
        alphas = 1.0 - betas
        self.register_buffer('alpha_bar', torch.cumprod(alphas, dim=0))

    def loss(self, actions, cond):
        B = actions.shape[0]
        actions_flat = actions.view(B, -1)
        t = torch.randint(0, self.T, (B,), device=actions.device)
        alpha_bar_t = self.alpha_bar[t].unsqueeze(-1)
        noise = torch.randn_like(actions_flat)
        noisy = torch.sqrt(alpha_bar_t) * actions_flat + torch.sqrt(1 - alpha_bar_t) * noise
        t_norm = t.float().unsqueeze(-1) / self.T
        inp = torch.cat([noisy, cond, t_norm], dim=-1)
        return F.mse_loss(self.noise_pred(inp), noise)

    @torch.no_grad()
    def sample(self, cond, n_steps=20):
        B = cond.shape[0]
        x = torch.randn(B, self.action_dim * self.horizon, device=cond.device)
        for i in reversed(range(min(n_steps, self.T))):
            t = torch.full((B,), i, device=cond.device)
            t_norm = t.float().unsqueeze(-1) / self.T
            inp = torch.cat([x, cond, t_norm], dim=-1)
            noise_pred = self.noise_pred(inp)
            alpha = 1 - (self.alpha_bar[i] if i > 0 else torch.tensor(1.0))
            alpha_bar_t = self.alpha_bar[i]
            x = (x - (1 - alpha) / torch.sqrt(1 - alpha_bar_t) * noise_pred) / torch.sqrt(1 - alpha + 1e-8)
            if i > 0:
                x = x + torch.sqrt(alpha) * torch.randn_like(x) * 0.1
        return x.view(B, self.horizon, self.action_dim)

print("All encoder + diffusion modules defined!")

In [ ]:
class MiniVLA(nn.Module):
    def __init__(self, vocab_size, state_dim, action_dim, horizon, d_model=128):
        super().__init__()
        self.img_encoder = ImageEncoderTinyCNN(d_model)
        self.txt_encoder = TextEncoderTinyGRU(vocab_size, d_word=64, d_model=d_model)
        self.state_encoder = StateEncoderMLP(state_dim, d_model)
        self.fusion = FusionMLP(d_model)
        self.diffusion_head = SimpleDiffusionHead(action_dim, horizon, cond_dim=d_model)

    def encode_obs(self, image, text_tokens, state):
        img_tok = self.img_encoder(image)
        txt_tok = self.txt_encoder(text_tokens)
        state_tok = self.state_encoder(state)
        return self.fusion(img_tok, txt_tok, state_tok)

    def loss(self, image, text_tokens, state, actions):
        cond = self.encode_obs(image, text_tokens, state)
        return self.diffusion_head.loss(actions, cond)

    def act(self, image, text_tokens, state):
        cond = self.encode_obs(image, text_tokens, state)
        return self.diffusion_head.sample(cond)

# Build the model
model = MiniVLA(VOCAB_SIZE, STATE_DIM, ACTION_DIM, HORIZON).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"mini-VLA created!")
print(f"  Total parameters: {total_params:,}")
print(f"  Vocab: {VOCAB_SIZE}, State: {STATE_DIM}-d (agent pos only), Action: {ACTION_DIM}-d x {HORIZON} steps")
print(f"\nParameter breakdown:")
for name, child in model.named_children():
    n = sum(p.numel() for p in child.parameters())
    print(f"  {name:>20s}: {n:>8,} params")

---
## Step 8: Train with Embedding Tracking

We train for 100 epochs and **snapshot the embedding weights** every 25 epochs. After training, we'll visualize how the direction words separate in embedding space.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 100
BATCH_SIZE = 32
N = data['images'].shape[0]

losses = []
embedding_snapshots = {}  # epoch -> weight matrix copy

# Save initial embeddings
embedding_snapshots[0] = model.txt_encoder.embed.weight.detach().cpu().clone()

model.train()
for epoch in range(EPOCHS):
    perm = torch.randperm(N)
    epoch_loss = 0
    n_batches = 0

    for i in range(0, N, BATCH_SIZE):
        idx = perm[i:i+BATCH_SIZE]
        imgs = data['images'][idx].to(device)
        txts = data['texts'][idx].to(device)
        sts = data['states'][idx].to(device)
        acts = data['actions'][idx].to(device)

        loss = model.loss(imgs, txts, sts, acts)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1

    avg_loss = epoch_loss / n_batches
    losses.append(avg_loss)

    # Snapshot embeddings every 25 epochs
    if (epoch + 1) % 25 == 0:
        embedding_snapshots[epoch + 1] = model.txt_encoder.embed.weight.detach().cpu().clone()
        # Print gradient magnitude on embedding layer
        grad_norm = model.txt_encoder.embed.weight.grad.norm().item() if model.txt_encoder.embed.weight.grad is not None else 0
        print(f"  Epoch {epoch+1:>3d}/{EPOCHS}: loss = {avg_loss:.4f}, embed grad norm = {grad_norm:.4f}")

print(f"\nTraining complete! Final loss: {losses[-1]:.4f}")
print(f"Embedding snapshots saved at epochs: {sorted(embedding_snapshots.keys())}")

In [ ]:
# Plot training curve
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(losses, color=ACCENT, linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Diffusion Loss (MSE)')
ax.set_title('mini-VLA Training Curve', fontsize=14, color=ACCENT, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Step 9: Visualize What Embeddings Learned

We use PCA to project the 64-d word embedding vectors down to 2D and see how they evolve during training.

**Prediction:** After training, the 4 direction words ("right", "left", "forward", "back") should **spread apart**, while the 3 function words ("push", "the", "block") should **cluster together** (they appear in every instruction and carry no directional information).

In [ ]:
from sklearn.decomposition import PCA

direction_words = ["right", "left", "forward", "back"]
function_words = ["push", "the", "block"]
all_plot_words = function_words + direction_words
dir_colors = {"right": ACCENT, "left": BLUE, "forward": TEAL, "back": PURPLE}
func_color = WARM

# PCA on epoch 0 vs epoch 100
epochs_to_show = [0, max(embedding_snapshots.keys())]
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, ep in zip(axes, epochs_to_show):
    weights = embedding_snapshots[ep]  # [vocab_size, 64]
    # Extract only the words we care about
    word_ids = [vocab[w] for w in all_plot_words]
    word_vecs = weights[word_ids].numpy()

    # PCA to 2D
    pca = PCA(n_components=2)
    coords = pca.fit_transform(word_vecs)

    # Plot function words
    for i, w in enumerate(all_plot_words):
        x, y = coords[i]
        c = dir_colors.get(w, func_color)
        marker = 'D' if w in direction_words else 'o'
        size = 120 if w in direction_words else 80
        ax.scatter(x, y, c=c, s=size, marker=marker, zorder=5, edgecolors='white', linewidth=1.5)
        ax.annotate(w, (x, y), textcoords="offset points", xytext=(8, 8),
                    fontsize=12, fontweight='bold', color=c)

    ax.set_title(f"Epoch {ep}", fontsize=14, color=ACCENT, fontweight='bold')
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%} var)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%} var)")
    ax.grid(True, alpha=0.3)

plt.suptitle("Word Embeddings: Before vs After Training",
             fontsize=16, color=ACCENT, fontweight='bold')
plt.tight_layout()
plt.show()

# Print cosine similarities after training
final_weights = embedding_snapshots[max(embedding_snapshots.keys())]
print("Cosine similarities AFTER training:")
for w1 in direction_words:
    for w2 in direction_words:
        if w1 < w2:
            v1 = final_weights[vocab[w1]]
            v2 = final_weights[vocab[w2]]
            cos = F.cosine_similarity(v1.unsqueeze(0), v2.unsqueeze(0)).item()
            print(f"  {w1:>8s} vs {w2:<8s}: {cos:+.3f}")

---
## Step 10: Test In-Distribution

We render a fresh PushT scene, run the model, and overlay the predicted trajectory on the real image. If training worked, the model should predict actions in the correct push direction.

In [ ]:
model.eval()

def test_on_env(model, instruction, seed=999):
    """Run model on a fresh PushT episode. Returns frame, predicted actions, expert actions, obs, seed."""
    env = gym.make("gym_pusht/PushT-v0", obs_type="state", render_mode="rgb_array")
    obs, info = env.reset(seed=seed)
    frame = env.render()
    goal_pose = info["goal_pose"]

    # Prepare model inputs
    img_pil = Image.fromarray(frame).resize((64, 64), Image.BILINEAR)
    img_tensor = torch.tensor(np.array(img_pil), dtype=torch.float32).permute(2, 0, 1) / 255.0
    tokens = torch.tensor([tokenize(instruction, vocab)])
    state = torch.tensor(obs[:STATE_DIM], dtype=torch.float32)

    # Model prediction
    with torch.no_grad():
        pred_actions = model.act(
            img_tensor.unsqueeze(0).to(device),
            tokens.to(device),
            state.unsqueeze(0).to(device)
        )
    pred = pred_actions[0].cpu().numpy()  # [8, 2]

    # Expert trajectory for comparison
    expert_actions = []
    obs_e, info_e = env.reset(seed=seed)
    for _ in range(HORIZON):
        a = expert_push_policy(obs_e, info_e["goal_pose"])
        expert_actions.append(a[:ACTION_DIM])
        obs_e, _, _, _, info_e = env.step(a)
    expert = np.array(expert_actions)

    env.close()
    return frame, pred, expert, obs

# Test all 4 directions — find seeds that match each direction
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
test_results = {}  # instr -> (frame, pred, expert, seed)

for col, instr in enumerate(["push the block right", "push the block left",
                               "push the block forward", "push the block back"]):
    # Find a seed matching this direction
    env_tmp = gym.make("gym_pusht/PushT-v0", obs_type="state")
    matched_seed = None
    for s in range(2000, 2500):
        obs_tmp, info_tmp = env_tmp.reset(seed=s)
        lbl = get_direction_label(obs_tmp[2:4], info_tmp["goal_pose"][:2])
        if lbl == instr:
            matched_seed = s
            break
    env_tmp.close()

    frame, pred, expert, obs = test_on_env(model, instr, seed=matched_seed)
    test_results[instr] = (frame, pred, expert, matched_seed)
    c = colors_map.get(instr, ACCENT)

    # Top row: rendered scene
    axes[0, col].imshow(frame)
    axes[0, col].set_title(f'"{instr}"', fontsize=11, color=c, fontweight='bold')
    axes[0, col].axis('off')

    # Bottom row: predicted vs expert waypoints
    axes[1, col].plot(expert[:, 0], expert[:, 1], 'o-', color=TEAL, linewidth=2,
                       markersize=5, label='Expert', alpha=0.7)
    axes[1, col].plot(pred[:, 0], pred[:, 1], 's--', color=c, linewidth=2,
                       markersize=5, label='Model')
    axes[1, col].legend(fontsize=9)
    axes[1, col].set_title(f"Waypoints: {instr.split()[-1]}", fontsize=11, color=c, fontweight='bold')
    axes[1, col].grid(True, alpha=0.3)
    axes[1, col].set_aspect('equal')

plt.suptitle("In-Distribution Test: Model vs Expert (4 directions)",
             fontsize=16, color=ACCENT, fontweight='bold')
plt.tight_layout()
plt.show()

# ── GIF rollouts: Expert vs Model side by side ──
print("\nRollout recordings — Expert (top label) vs Model (bottom label):")
html_parts = ['<div style="display:flex; flex-wrap:wrap; gap:16px; justify-content:center; margin-top:8px;">']

for instr in ["push the block right", "push the block left",
              "push the block forward", "push the block back"]:
    frame, pred, expert, sd = test_results[instr]
    c = colors_map[instr]
    direction = instr.split()[-1]

    # Expert rollout
    expert_frames = rollout_waypoints(expert, seed=sd)
    expert_b64 = frames_to_gif_b64(expert_frames)

    # Model rollout
    model_frames = rollout_waypoints(pred, seed=sd)
    model_b64 = frames_to_gif_b64(model_frames)

    html_parts.append(f'''
    <div style="text-align:center; border:1px solid #e5ddd3; border-radius:12px; padding:10px; background:#faf8f5;">
      <div style="color:{c}; font-weight:bold; font-size:14px; margin-bottom:6px;">"{instr}"</div>
      <div style="display:flex; gap:8px; justify-content:center;">
        <div>
          <div style="font-size:11px; color:{TEAL}; font-weight:bold; margin-bottom:3px;">Expert</div>
          <img src="data:image/gif;base64,{expert_b64}" style="width:180px; border:2px solid {TEAL}; border-radius:6px;"/>
        </div>
        <div>
          <div style="font-size:11px; color:{c}; font-weight:bold; margin-bottom:3px;">Model</div>
          <img src="data:image/gif;base64,{model_b64}" style="width:180px; border:2px solid {c}; border-radius:6px;"/>
        </div>
      </div>
    </div>''')

html_parts.append('</div>')
display(HTML(''.join(html_parts)))
print("Compare: the model's predicted waypoints should push in roughly the same direction as the expert.")

---
## Step 11: Four Failure Tests

Now the important part — where does this mini-VLA **break**? We test 4 scenarios that expose fundamental limitations of training from scratch.

In [ ]:
# Find a test seed for "push the block right"
env_tmp = gym.make("gym_pusht/PushT-v0", obs_type="state")
test_seed = None
for s in range(3000, 3500):
    obs_tmp, info_tmp = env_tmp.reset(seed=s)
    if get_direction_label(obs_tmp[2:4], info_tmp["goal_pose"][:2]) == "push the block right":
        test_seed = s
        break
env_tmp.close()

# Ground truth: in-distribution test
frame_gt, pred_gt, expert_gt, obs_gt = test_on_env(model, "push the block right", seed=test_seed)

# ── FAILURE 1: Synonym instruction ("move the cube rightward") ──
_, pred_syn, _, _ = test_on_env(model, "move the cube rightward", seed=test_seed)

# ── FAILURE 2: Random noise image ──
noise_img = torch.randn(3, 64, 64) * 0.3
tokens_f2 = torch.tensor([tokenize("push the block right", vocab)])
state_f2 = torch.tensor(obs_gt[:STATE_DIM], dtype=torch.float32)
with torch.no_grad():
    pred_noise = model.act(
        noise_img.unsqueeze(0).to(device),
        tokens_f2.to(device),
        state_f2.unsqueeze(0).to(device)
    )[0].cpu().numpy()

# ── FAILURE 3: Completely new task ──
_, pred_newtask, _, _ = test_on_env(model, "pick up the block", seed=test_seed)

# ── Static plots: all 4 side by side ──
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

tests = [
    ("In-Distribution", "push the block right", pred_gt, TEAL, True),
    ("Synonym Words", "move the cube rightward", pred_syn, RED, False),
    ("Noise Image", "push the block right", pred_noise, RED, False),
    ("New Task", "pick up the block", pred_newtask, RED, False),
]

for col, (title, instr, pred, c, ok) in enumerate(tests):
    if title == "Noise Image":
        noise_vis = (noise_img.permute(1, 2, 0).numpy() * 0.5 + 0.5).clip(0, 1)
        axes[0, col].imshow(noise_vis)
    else:
        axes[0, col].imshow(frame_gt)
    status = "PASS" if ok else "FAIL"
    axes[0, col].set_title(f'{title} ({status})', fontsize=12, color=c, fontweight='bold')
    axes[0, col].set_xlabel(f'"{instr}"', fontsize=10)
    axes[0, col].set_xticks([]); axes[0, col].set_yticks([])

    axes[1, col].plot(expert_gt[:, 0], expert_gt[:, 1], 'o-', color=TEAL, linewidth=2,
                       markersize=5, label='Expert', alpha=0.5)
    axes[1, col].plot(pred[:, 0], pred[:, 1], 's--', color=c, linewidth=2,
                       markersize=5, label='Model')
    axes[1, col].legend(fontsize=9)
    axes[1, col].grid(True, alpha=0.3)
    axes[1, col].set_aspect('equal')

plt.suptitle("Generalization Tests: 1 Pass, 3 Fails",
             fontsize=16, color=ACCENT, fontweight='bold')
plt.tight_layout()
plt.show()

# ── GIF rollouts for all 4 failure tests ──
print("\nRollout recordings — watch the robot execute each test:")
html_parts = ['<div style="display:flex; flex-wrap:wrap; gap:16px; justify-content:center; margin-top:8px;">']

# Expert rollout (ground truth)
expert_frames = rollout_waypoints(expert_gt, seed=test_seed)
expert_b64 = frames_to_gif_b64(expert_frames)

failure_tests_for_gif = [
    ("PASS: In-Distribution", "push the block right", pred_gt, TEAL),
    ("FAIL: Synonym Words", "move the cube rightward", pred_syn, RED),
    ("FAIL: Noise Image", "push the block right", pred_noise, RED),
    ("FAIL: New Task", "pick up the block", pred_newtask, RED),
]

for title, instr, pred, c in failure_tests_for_gif:
    # Model rollout with its predicted waypoints
    model_frames = rollout_waypoints(pred, seed=test_seed)
    model_b64 = frames_to_gif_b64(model_frames)

    ok = "PASS" in title
    border_c = TEAL if ok else RED
    bg = "#f0f7f2" if ok else "#fdf0ee"

    html_parts.append(f'''
    <div style="text-align:center; border:2px solid {border_c}; border-radius:12px; padding:10px; background:{bg}; min-width:200px;">
      <div style="color:{border_c}; font-weight:bold; font-size:13px; margin-bottom:2px;">{title}</div>
      <div style="font-size:11px; color:#8c7e6f; margin-bottom:6px;">"{instr}"</div>
      <div style="display:flex; gap:6px; justify-content:center;">
        <div>
          <div style="font-size:10px; color:{TEAL}; font-weight:bold;">Expert</div>
          <img src="data:image/gif;base64,{expert_b64}" style="width:150px; border:1px solid {TEAL}; border-radius:4px;"/>
        </div>
        <div>
          <div style="font-size:10px; color:{border_c}; font-weight:bold;">Model</div>
          <img src="data:image/gif;base64,{model_b64}" style="width:150px; border:1px solid {border_c}; border-radius:4px;"/>
        </div>
      </div>
    </div>''')

html_parts.append('</div>')
display(HTML(''.join(html_parts)))

print("\nResults:")
print("  1. In-Distribution:  PASS — model pushes in the correct direction (matches expert)")
print("  2. Synonym Words:    FAIL — 'move', 'cube', 'rightward' are UNKNOWN tokens (mapped to <pad>)")
print("  3. Noise Image:      FAIL — CNN has never seen random noise, outputs garbage actions")
print("  4. New Task:         FAIL — 'pick', 'up' are UNKNOWN; model has no concept of picking")

---
## Summary: Why the mini-VLA Fails

| Test | Result | Root Cause | What Would Fix It |
|------|--------|-----------|-------------------|
| In-distribution ("push the block right") | PASS | Training data covers this case | — |
| Synonym ("move the cube rightward") | FAIL | Unknown words map to `<pad>` — GRU trained on only 7 words | Pre-trained language model (trillions of tokens) |
| Different visual input (noise image) | FAIL | TinyCNN trained on 200 PushT frames only | Pre-trained vision encoder (billions of images) |
| New task ("pick up the block") | FAIL | Unknown verb + no concept of lifting in training data | Transformer with cross-attention + diverse training tasks |

**The architecture is correct** — encode vision + language + state, fuse them, generate actions. But learning all three modalities from scratch with 200 episodes is impossible.

Modern VLAs like pi0, RT-2, and OpenVLA solve this by plugging in **pre-trained foundation models**:
- **SigLIP/ViT** for vision (trained on billions of image-text pairs)
- **PaliGemma/LLaMA** for language (trained on trillions of tokens)
- **Transformer cross-attention** for fusion (instead of MLP concatenation)

The pre-trained models provide "world knowledge" for free — they already know that "move" = "push" and what a T-block looks like from any angle.

---

## Exercises

1. **More data:** Increase to 1000 episodes. Does the synonym test start working? (Hint: no — it's a vocabulary problem, not a data problem)
2. **Deeper fusion:** Replace `FusionMLP` with a 4-layer MLP. Does cross-modal reasoning improve?
3. **Pre-trained vision:** Replace `TinyCNN` with `torchvision.models.resnet18(pretrained=True)`. Does the noise-image test improve?
4. **Challenge:** Add 4 more instructions ("push the block up-right", "push the block up-left", etc.). Can the model learn 8 directions with the same 200 episodes?